# Zadanie: model 1 — regresja `streams`

**Pytanie badawcze:** jakie cechy utworu wpływają na liczbę odtworzeń?

**Jak pracować z notebookiem**

- Uruchamiaj komórki po kolei, od góry.
- W komórkach oznaczonych **✏️ Zadanie** uzupełnij miejsca `___`. Pozostałe komórki uruchom bez zmian.
- Pod każdym **❓ Pytaniem** dopisz krótką odpowiedź w komórce *Odpowiedź*.
- Dane wczytują się z folderu `dane/` repozytorium, a jeśli go nie ma (np. w Colabie), bezpośrednio z GitHuba.
- Rozwiązanie: `gotowe-notebooki/01_regresja_streams.ipynb`.

## 1. Przygotowanie

### Importy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm

from sklearn import model_selection, linear_model, metrics, ensemble, dummy
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

### ✏️ Zadanie 1: wczytanie danych i wybór zmiennych

Ustaw zmienną celu i zbuduj macierz cech `X` (cechy numeryczne + gatunek) oraz wektor `y`.

In [ ]:
from pathlib import Path

URL = "https://raw.githubusercontent.com/mzntk/ml-seminarium-projekt/main/dane/viral_songs.csv"
LOCAL = Path("../dane/viral_songs.csv")   # ścieżka w sklonowanym repozytorium

df = pd.read_csv(LOCAL if LOCAL.exists() else URL)

NUM_FEATURES = [
    "release_year", "duration_sec", "danceability", "energy", "valence", "tempo_bpm",
    "acousticness", "instrumentalness", "speechiness", "liveness", "loudness_db",
]
CAT_FEATURES = ["genre"]
TARGET = ___   # zmienna objaśniana

X = df[___]
y = df[___]
print(X.shape, y.shape)

**❓ Pytanie:** Dlaczego wśród cech nie ma kolumny `viral`? (Wskazówka: wróć do EDA, zadanie 11.)

**Odpowiedź:** …

### ✏️ Zadanie 2: podział na zbiór treningowy i testowy

70% danych do uczenia, 30% do testu.

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, train_size=___, random_state=42
)
print("train:", X_train.shape, "test:", X_test.shape)

### ✏️ Zadanie 3: przygotowanie cech i szablon modelu

- `StandardScaler`: każda cecha numeryczna ma średnią 0 i odchylenie standardowe 1
- `OneHotEncoder(drop="first")`: zamiana `genre` na 9 zmiennych 0/1, bo model nie czyta słów
- `make_model`: model uczy się na `log10(streams)`, a predykcje zwraca w oryginalnej skali `streams`

In [ ]:
preprocess = ColumnTransformer([
    ("num", ___, NUM_FEATURES),
    ("cat", ___, CAT_FEATURES),
])

def make_model(regressor):
    pipe = Pipeline([("prep", preprocess), ("reg", regressor)])
    return TransformedTargetRegressor(
        regressor=pipe, func=___, inverse_func=___
    )

**❓ Pytanie:** Dlaczego przed Ridge i LASSO cechy trzeba standaryzować? Po co `drop="first"` w kodowaniu gatunku?

**Odpowiedź:** …

### Funkcje pomocnicze (z podręcznika)

Uruchom bez zmian.

In [ ]:
def sse(resid):
    return np.sum(resid ** 2)

def r2_log(y_true, y_pred):
    return metrics.r2_score(np.log10(y_true), np.log10(y_pred))

def evaluate(model, X_tr, y_tr, X_te, y_te):
    pred_tr, pred_te = model.predict(X_tr), model.predict(X_te)
    return {
        "R2_log_train": r2_log(y_tr, pred_tr),
        "R2_log_test": r2_log(y_te, pred_te),
        "R2_test": metrics.r2_score(y_te, pred_te),
        "MAE_test": metrics.mean_absolute_error(y_te, pred_te),
        "MedAE_test": metrics.median_absolute_error(y_te, pred_te),
    }

def feature_names(model):
    return model.regressor_.named_steps["prep"].get_feature_names_out()

def plot_residuals_and_coeff(resid_train, resid_test, coeff, names, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].bar(np.arange(len(resid_train)), resid_train)
    axes[0].set_xlabel("numer próbki")
    axes[0].set_ylabel("reszta")
    axes[0].set_title("zbiór treningowy")
    axes[1].bar(np.arange(len(resid_test)), resid_test)
    axes[1].set_xlabel("numer próbki")
    axes[1].set_ylabel("reszta")
    axes[1].set_title("zbiór testowy")
    axes[2].bar(np.arange(len(coeff)), coeff)
    axes[2].set_xticks(np.arange(len(coeff)), names, rotation=90)
    axes[2].set_ylabel("współczynnik")
    axes[2].set_title("współczynniki")
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes

### Funkcja pomocnicza: ważność cech jak `varImp` w R

Uruchom bez zmian.

In [ ]:
def var_imp(values, title, signed=None):
    values = pd.Series(values)
    rng = values.max() - values.min()
    overall = 100 * (values - values.min()) / rng if rng > 0 else values * 0
    table = pd.DataFrame({"Overall": overall})
    if signed is not None:
        table.insert(0, "współczynnik", signed)
    table = table.sort_values("Overall", ascending=False)
    display(table.round(4))

    colors = "grey"
    if signed is not None:
        colors = np.where(table["współczynnik"] >= 0, "tab:blue", "tab:red")[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(table) + 1))
    ax.barh(table.index[::-1], table["Overall"][::-1], color=colors)
    ax.set_xlabel("ważność (0–100)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## 2. Model bazowy

### ✏️ Zadanie 4

`DummyRegressor` ignoruje cechy i dla każdego utworu przewiduje tę samą wartość: średnią z `log10(streams)` w zbiorze treningowym. To punkt odniesienia — każdy model powinien być od niego lepszy.

In [ ]:
baseline = make_model(dummy.DummyRegressor(strategy=___))
baseline.fit(X_train, y_train)
results = {"Dummy (średnia)": evaluate(baseline, X_train, y_train, X_test, y_test)}
pd.DataFrame(results).T

**❓ Pytanie:** Dlaczego `R2_log_test` modelu bazowego jest bliskie 0?

**Odpowiedź:** …

## 3. Regresja liniowa

### ✏️ Zadanie 5: dopasowanie modelu

In [ ]:
lin = make_model(___)
lin.___
results["LinearRegression"] = evaluate(___, X_train, y_train, X_test, y_test)
pd.DataFrame(results).T

### ✏️ Zadanie 6: reszty i współczynniki

Wyciągnij współczynniki z kroku `"reg"` pipeline’u i policz reszty (wartość rzeczywista − predykcja) na obu zbiorach.

In [ ]:
coef = lin.regressor_.named_steps[___].coef_
resid_train = y_train - ___
resid_test = y_test - ___
fig, ax = plot_residuals_and_coeff(resid_train, resid_test, coef, feature_names(lin), "LinearRegression")
plt.show()

### ✏️ Zadanie 7: summary i ważność cech

Dopasuj ten sam model w `statsmodels`, żeby zobaczyć błędy standardowe, statystyki t i p-wartości. Zmienna objaśniana to `log10(streams)`.

In [ ]:
prep = lin.regressor_.named_steps["prep"]
X_train_prep = pd.DataFrame(prep.transform(X_train), columns=feature_names(lin), index=X_train.index)

ols = sm.OLS(___, sm.add_constant(X_train_prep)).fit()
print(ols.summary())

var_imp(ols.tvalues.drop("const").abs(), "LinearRegression — ważność (|t|)",
        signed=ols.params.drop("const"))

**❓ Pytanie:** Które cechy są istotne statystycznie (p < 0,05)? Jak zinterpretować współczynnik przy `danceability`? (Wskazówka: cechy są standaryzowane, a zmienna objaśniana jest w skali log₁₀ — wzrost cechy o 1 odchylenie standardowe mnoży liczbę odtworzeń przez 10^β.)

**Odpowiedź:** …

## 4. Regresja Ridge

### ✏️ Zadanie 8

Dopasuj regresję Ridge z α = 2,5.

In [ ]:
ridge = make_model(linear_model.___(alpha=___))
ridge.fit(X_train, y_train)
results["Ridge (α = 2.5)"] = evaluate(ridge, X_train, y_train, X_test, y_test)
pd.DataFrame(results).T

### ✏️ Zadanie 9: reszty i współczynniki

Na wzór zadania 6.

In [ ]:
coef = ___
resid_train = ___
resid_test = ___
fig, ax = plot_residuals_and_coeff(resid_train, resid_test, coef, feature_names(ridge), "Ridge (α = 2.5)")
plt.show()

### Ważność cech

In [ ]:
var_imp(np.abs(pd.Series(ridge.regressor_.named_steps["reg"].coef_, index=feature_names(ridge))),
        "Ridge (α = 2.5) — ważność (|β|)",
        signed=ridge.regressor_.named_steps["reg"].coef_)

## 5. Regresja LASSO

### ✏️ Zadanie 10

Dopasuj regresję LASSO z α = 0,01.

In [ ]:
lasso = make_model(linear_model.___(alpha=___))
lasso.fit(X_train, y_train)
results["Lasso (α = 0.01)"] = evaluate(lasso, X_train, y_train, X_test, y_test)
pd.DataFrame(results).T

### Reszty i współczynniki

In [ ]:
coef = lasso.regressor_.named_steps["reg"].coef_
resid_train = y_train - lasso.predict(X_train)
resid_test = y_test - lasso.predict(X_test)
fig, ax = plot_residuals_and_coeff(resid_train, resid_test, coef, feature_names(lasso), "Lasso (α = 0.01)")
plt.show()

### Ważność cech

In [ ]:
var_imp(np.abs(pd.Series(lasso.regressor_.named_steps["reg"].coef_, index=feature_names(lasso))),
        "Lasso (α = 0.01) — ważność (|β|)",
        signed=lasso.regressor_.named_steps["reg"].coef_)

**❓ Pytanie:** Które współczynniki LASSO wyzerował? Porównaj je ze współczynnikami regresji liniowej i Ridge.

**Odpowiedź:** …

## 6. Wpływ α na model: ścieżka LASSO

### ✏️ Zadanie 11: pętla po α

Dla 50 wartości α z przedziału [10⁻⁴, 10⁰] (skala logarytmiczna) dopasuj LASSO i zapisz współczynniki oraz SSE na zbiorze treningowym i testowym.

In [ ]:
alphas = np.___(-4, 0, 50)
names = feature_names(lasso)
coeffs = np.zeros((len(alphas), len(names)))
sse_train = np.zeros_like(alphas)
sse_test = np.zeros_like(alphas)

for n, alpha in enumerate(alphas):
    model = make_model(linear_model.Lasso(alpha=___))
    model.___
    coeffs[n, :] = ___
    sse_train[n] = sse(np.log10(y_train) - np.log10(model.predict(X_train)))
    sse_test[n] = sse(np.log10(y_test) - np.log10(model.predict(X_test)))

### Wykres ścieżki LASSO

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for i, name in enumerate(names):
    axes[0].plot(np.log10(alphas), coeffs[:, i], lw=1, label=name)
axes[0].set_xlabel(r"$\log_{10}\alpha$")
axes[0].set_ylabel("współczynniki")
axes[0].legend(fontsize=7, ncol=2)

axes[1].plot(np.log10(alphas), sse_train, label="train")
axes[1].plot(np.log10(alphas), sse_test, label="test")
axes[1].set_xlabel(r"$\log_{10}\alpha$")
axes[1].set_ylabel("SSE")
axes[1].legend()
plt.tight_layout()
plt.show()

**❓ Pytanie:** Przy jakim α większość współczynników spada do zera? Które cechy „przeżywają” najdłużej? Czy na wykresie SSE widać przeuczenie przy małych α, jak w przykładzie z podręcznika? Z czego może wynikać różnica?

**Odpowiedź:** …

## 7. Dobór α przez walidację krzyżową

### ✏️ Zadanie 12: RidgeCV

Sprawdź 100 wartości α z przedziału [10⁻⁴, 10⁴] w 5-krotnej walidacji krzyżowej i wypisz wybrane α.

In [ ]:
ridge_cv = make_model(linear_model.RidgeCV(alphas=np.logspace(-4, 4, 100), cv=___))
ridge_cv.fit(X_train, y_train)
print("RidgeCV α =", ridge_cv.regressor_.named_steps["reg"].___)
results["RidgeCV"] = evaluate(ridge_cv, X_train, y_train, X_test, y_test)
display(pd.DataFrame(results).T)

In [ ]:
coef = ridge_cv.regressor_.named_steps["reg"].coef_
resid_train = y_train - ridge_cv.predict(X_train)
resid_test = y_test - ridge_cv.predict(X_test)
fig, ax = plot_residuals_and_coeff(resid_train, resid_test, coef, feature_names(ridge_cv), "RidgeCV")
plt.show()

In [ ]:
var_imp(np.abs(pd.Series(ridge_cv.regressor_.named_steps["reg"].coef_, index=feature_names(ridge_cv))),
        "RidgeCV — ważność (|β|)",
        signed=ridge_cv.regressor_.named_steps["reg"].coef_)

### ✏️ Zadanie 13: LassoCV

In [ ]:
lasso_cv = make_model(linear_model.___(cv=5, random_state=42))
lasso_cv.fit(X_train, y_train)
print("LassoCV α =", lasso_cv.regressor_.named_steps["reg"].alpha_)
results["LassoCV"] = evaluate(lasso_cv, X_train, y_train, X_test, y_test)
display(pd.DataFrame(results).T)

In [ ]:
coef = lasso_cv.regressor_.named_steps["reg"].coef_
resid_train = y_train - lasso_cv.predict(X_train)
resid_test = y_test - lasso_cv.predict(X_test)
fig, ax = plot_residuals_and_coeff(resid_train, resid_test, coef, feature_names(lasso_cv), "LassoCV")
plt.show()

In [ ]:
var_imp(np.abs(pd.Series(lasso_cv.regressor_.named_steps["reg"].coef_, index=feature_names(lasso_cv))),
        "LassoCV — ważność (|β|)",
        signed=lasso_cv.regressor_.named_steps["reg"].coef_)

### ✏️ Zadanie 14: ElasticNetCV

Elastic Net łączy kary L1 i L2. Parametr `l1_ratio` określa udział kary L1: 0 to czysty Ridge, 1 to czysty LASSO. Sprawdź `l1_ratio` ∈ {0,1; 0,5; 0,7; 0,9; 0,95; 1} i wypisz wybrane α oraz `l1_ratio`.

In [ ]:
enet_cv = make_model(linear_model.ElasticNetCV(l1_ratio=___, cv=5, random_state=42))
enet_cv.fit(X_train, y_train)
print("ElasticNetCV α =", enet_cv.regressor_.named_steps["reg"].alpha_,
      " l1_ratio =", enet_cv.regressor_.named_steps["reg"].___)
results["ElasticNetCV"] = evaluate(enet_cv, X_train, y_train, X_test, y_test)
display(pd.DataFrame(results).T)

In [ ]:
coef = enet_cv.regressor_.named_steps["reg"].coef_
resid_train = y_train - enet_cv.predict(X_train)
resid_test = y_test - enet_cv.predict(X_test)
fig, ax = plot_residuals_and_coeff(resid_train, resid_test, coef, feature_names(enet_cv), "ElasticNetCV")
plt.show()

In [ ]:
var_imp(np.abs(pd.Series(enet_cv.regressor_.named_steps["reg"].coef_, index=feature_names(enet_cv))),
        "ElasticNetCV — ważność (|β|)",
        signed=enet_cv.regressor_.named_steps["reg"].coef_)

**❓ Pytanie:** Jakie α wybrała walidacja krzyżowa dla każdego modelu? Czy regularyzacja poprawiła wynik na zbiorze testowym względem zwykłej regresji liniowej?

**Odpowiedź:** …

## 8. Random Forest

Model nieliniowy: średnia predykcji wielu drzew decyzyjnych uczonych na losowych podpróbach danych i cech. Może wychwycić interakcje i nieliniowości, których modele liniowe nie widzą.

### ✏️ Zadanie 15

Dopasuj las z 300 drzewami i minimum 5 obserwacjami w liściu (`n_estimators`, `min_samples_leaf`).

In [ ]:
rf = make_model(ensemble.RandomForestRegressor(n_estimators=___, min_samples_leaf=___, n_jobs=-1, random_state=42))
rf.fit(X_train, y_train)
results["RandomForest"] = evaluate(rf, X_train, y_train, X_test, y_test)
pd.DataFrame(results).T

### ✏️ Zadanie 16: ważność cech

Permutation importance: spadek R² na zbiorze testowym po wymieszaniu wartości danej cechy. Kolumnę jednej cechy w zbiorze testowym losowo przestawiamy między wierszami; pozostałe kolumny i zmienna celu zostają na swoich miejscach. Podaj model i **testowe** dane.

In [ ]:
var_imp(pd.Series(rf.regressor_.named_steps["reg"].feature_importances_, index=feature_names(rf)),
        "RandomForest — ważność (spadek zanieczyszczenia, ~IncNodePurity)")

perm = permutation_importance(___, n_repeats=10, random_state=42, n_jobs=-1,
                              scoring=metrics.make_scorer(r2_log))
var_imp(pd.Series(perm.importances_mean, index=X_test.columns),
        "RandomForest — ważność permutacyjna (~%IncMSE)")

**❓ Pytanie:** Porównaj `R2_log_train` i `R2_log_test` lasu. Czy model jest przeuczony? Czy jest lepszy od modeli liniowych?

**Odpowiedź:** …

## 9. Porównanie modeli

### Tabela porównawcza

In [ ]:
res = pd.DataFrame(results).T
display(res.style.format({
    "R2_log_train": "{:.3f}", "R2_log_test": "{:.3f}", "R2_test": "{:.3f}",
    "MAE_test": "{:,.0f}", "MedAE_test": "{:,.0f}",
}))

### Wykres porównawczy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
res[["R2_log_train", "R2_log_test"]].plot.bar(ax=axes[0])
axes[0].set_title("R² (log10 streams)")
axes[0].tick_params(axis="x", rotation=45)
res["MedAE_test"].plot.bar(ax=axes[1])
axes[1].set_title("Mediana błędu bezwzględnego (test)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Wartości rzeczywiste a przewidywane

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, [("LassoCV", lasso_cv), ("RandomForest", rf)]):
    pred = model.predict(X_test)
    ax.scatter(y_test, pred, s=5, alpha=0.3)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, color="red", lw=1)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("streams rzeczywiste")
    ax.set_ylabel("streams przewidywane")
    ax.set_title(name)
plt.tight_layout()
plt.show()

**❓ Pytanie:** Który model wypada najlepiej? Dlaczego `R2_test` (skala oryginalna) jest tak różne od `R2_log_test`? Co wykres rzeczywiste vs przewidywane mówi o zakresie predykcji?

**Odpowiedź:** …

## 10. Ważność cech — wszystkie modele

### Współczynniki modeli liniowych

In [ ]:
linear_models = {
    "LinearRegression": lin,
    "Ridge (α = 2.5)": ridge,
    "Lasso (α = 0.01)": lasso,
    "RidgeCV": ridge_cv,
    "LassoCV": lasso_cv,
    "ElasticNetCV": enet_cv,
}

coef_table = pd.DataFrame({
    name: model.regressor_.named_steps["reg"].coef_
    for name, model in linear_models.items()
}, index=feature_names(lin))

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(coef_table, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Współczynniki modeli liniowych (cechy standaryzowane)")
plt.tight_layout()
plt.show()

### Permutation importance

In [ ]:
all_models = {**linear_models, "RandomForest": rf}

perm_table = pd.DataFrame({
    name: permutation_importance(
        model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1,
        scoring=metrics.make_scorer(r2_log),
    ).importances_mean
    for name, model in all_models.items()
}, index=X_test.columns)

perm_table = perm_table.loc[perm_table.mean(axis=1).sort_values(ascending=False).index]
display(perm_table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(perm_table, annot=True, fmt=".3f", cmap="Reds", ax=ax)
ax.set_title("Permutation importance według modelu (spadek R² log)")
plt.tight_layout()
plt.show()

## 11. Wnioski

Odpowiedz na pytanie badawcze: jakie cechy utworu wpływają na liczbę odtworzeń? Czy wszystkie modele wskazują te same cechy? Jak dobrze cechy audio i gatunek wyjaśniają `streams` i co może odpowiadać za resztę zmienności?

**Odpowiedź:** …